In [1]:
from sympy import *
from sympy.abc import *
from sympy.vector import CoordSys3D, Del
from sympy import I # imaginary unit

# Уравнения Максвелла

In [2]:
R = CoordSys3D('')
delop = Del()
phi = Function('phi')
Ex, Ey, Ez, E = symbols('E_x E_y E_z E', cls=Function)
Hx, Hy, Hz, H = symbols('H_x, H_y, H_z, H', cls=Function)

In [3]:
E_field = Ex(R.x,R.y,R.z,t)*R.i + Ey(R.x,R.y,R.z,t)*R.j + Hz(R.x,R.y,R.z,t)*R.k
H_field = Hx(R.x,R.y,R.z,t)*R.i + Hy(R.x,R.y,R.z,t)*R.j + Hz(R.x,R.y,R.z,t)*R.k

In [4]:
( delop.cross(E_field) + mu/c*H_field.diff(t) ).to_matrix(R)

Matrix([
[-Derivative(E_y(.x, .y, .z, t), .z) + Derivative(H_z(.x, .y, .z, t), .y) + mu*Derivative(H_x(.x, .y, .z, t), t)/c],
[ Derivative(E_x(.x, .y, .z, t), .z) - Derivative(H_z(.x, .y, .z, t), .x) + mu*Derivative(H_y(.x, .y, .z, t), t)/c],
[-Derivative(E_x(.x, .y, .z, t), .y) + Derivative(E_y(.x, .y, .z, t), .x) + mu*Derivative(H_z(.x, .y, .z, t), t)/c]])

In [957]:
( delop.cross(H_field) - epsilon/c*E_field.diff(t) ).to_matrix(R)

Matrix([
[-Derivative(H_y(.x, .y, .z, t), .z) + Derivative(H_z(.x, .y, .z, t), .y) - epsilon*Derivative(E_x(.x, .y, .z, t), t)/c],
[ Derivative(H_x(.x, .y, .z, t), .z) - Derivative(H_z(.x, .y, .z, t), .x) - epsilon*Derivative(E_y(.x, .y, .z, t), t)/c],
[-Derivative(H_x(.x, .y, .z, t), .y) + Derivative(H_y(.x, .y, .z, t), .x) - epsilon*Derivative(H_z(.x, .y, .z, t), t)/c]])

# Введение асимптотического разложения

In [1072]:
s=1 # expansion order
E_symbols = [[Function(f"E^{comp}_{i}") for comp in [x,y,z]] for i in range(s+1)]
E_vec_comps = [Ex(R.x,R.y,R.z)*R.i + Ey(R.x,R.y,R.z)*R.j + Ez(R.x,R.y,R.z)*R.k for Ex, Ey, Ez in E_symbols]
H_symbols = [[Function(f"H^{comp}_{i}") for comp in [x,y,z]] for i in range(s+1)]
H_vec_comps = [Hx(R.x,R.y,R.z)*R.i + Hy(R.x,R.y,R.z)*R.j + Hz(R.x,R.y,R.z)*R.k for Hx, Hy, Hz in H_symbols]
E_asympt = sum(map(lambda k, vec: vec/(I*omega)**k,range(s+1),E_vec_comps), 0*R.i) # 0*R.i determines return type
H_asympt = sum(map(lambda k, vec: vec/(I*omega)**k,range(s+1),H_vec_comps), 0*R.i)

In [13]:
E_asympt[0].to_matrix(R)

Matrix([
[E^x_0(.x, .y, .z)],
[E^y_0(.x, .y, .z)],
[E^z_0(.x, .y, .z)]])

In [1052]:
def gen_subs_diff(U_comps):
    from itertools import product
    return {value.diff(key):0 for key, value 
            in list(product([R.y, R.z], list(U_comps.components.values())))}

curl_E = (( delop.cross(E_asympt*exp(I*omega*t - I*omega/c*phi(R.z))) + 
          mu/c*(H_asympt*exp(I*omega*t - I*omega/c*phi(R.z))).diff(t)).doit().to_matrix(R)
               / exp(I*omega*t - I*omega/c*phi(R.z))).expand().subs(gen_subs_diff(E_vec_comps[0]))

curl_H = (( delop.cross(H_asympt*exp(I*omega*t - I*omega/c*phi(R.z))) + 
     epsilon/c*(E_asympt*exp(I*omega*t - I*omega/c*phi(R.z))).diff(t)).doit().to_matrix(R)
               / exp(I*omega*t - I*omega/c*phi(R.z))).expand().subs(gen_subs_diff(H_vec_comps[0]))

In [1054]:
gen_subs_diff(H_vec_comps[0])

{Derivative(H^x_0(.x, .y, .z), .y): 0,
 Derivative(H^y_0(.x, .y, .z), .y): 0,
 Derivative(H^z_0(.x, .y, .z), .y): 0,
 Derivative(H^x_0(.x, .y, .z), .z): 0,
 Derivative(H^y_0(.x, .y, .z), .z): 0,
 Derivative(H^z_0(.x, .y, .z), .z): 0}

In [1055]:
curl_E

Matrix([
[                                     I*mu*omega*H^x_0(.x, .y, .z)/c + I*omega*E^y_0(.x, .y, .z)*Derivative(phi(.z), .z)/c],
[-Derivative(E^z_0(.x, .y, .z), .x) + I*mu*omega*H^y_0(.x, .y, .z)/c - I*omega*E^x_0(.x, .y, .z)*Derivative(phi(.z), .z)/c],
[                                                       Derivative(E^y_0(.x, .y, .z), .x) + I*mu*omega*H^z_0(.x, .y, .z)/c]])

In [1056]:
maxwell_eqs=[Eq(eq,0) for eq in list(curl_E)+list(curl_H)]

def list_subs(eqs, subs):
    return list(map(lambda eq: eq.subs(subs),eqs))

In [1057]:
maxwell_eqs = list_subs(maxwell_eqs, {R.x:x,R.y:y,R.z:z})

In [1058]:
maxwell_eqs[2]

Eq(Derivative(E^y_0(x, y, z), x) + I*mu*omega*H^z_0(x, y, z)/c, 0)

In [1059]:
maxwell_diff_eqs[1]

Eq(Derivative(E^y_0(x), x) + I*mu*omega*H^z_0(x)/c, 0)

In [1060]:
maxwell_eqs[1].subs(maxwell_alg_eqs)

Eq(-Derivative(E^z_0(x, y, z), x) + I*mu*omega*H^y_0(x, y, z)/c + I*omega*H^y_0(x, y, z)*Derivative(phi(z), z)**2/(c*epsilon), 0)

In [1061]:
maxwell_alg_eqs = {H_symbols[0][0](x,y,z): solve(maxwell_eqs[0],H_symbols[0][0](x,y,z))[0],
               E_symbols[0][0](x,y,z): solve(maxwell_eqs[3],E_symbols[0][0](x,y,z))[0]}

In [1062]:
maxwell_alg

{H^x_0(x, y, z): -E^y_0(x, y, z)*Derivative(phi(z), z)/mu,
 E^x_0(x, y, z): -H^y_0(x, y, z)*Derivative(phi(z), z)/epsilon}

In [1063]:
maxwell_diff_eqs = list(filter(lambda x: x is not sympify(True),list_subs(maxwell_eqs, maxwell_alg_eqs)))

In [1065]:
maxwell_diff_eqs[0]

Eq(-Derivative(E^z_0(x, y, z), x) + I*mu*omega*H^y_0(x, y, z)/c + I*omega*H^y_0(x, y, z)*Derivative(phi(z), z)**2/(c*epsilon), 0)

In [1064]:
H_symbols

[[H^x_0, H^y_0, H^z_0]]

In [1066]:
from sympy.solvers.ode.systems import dsolve_system
def gen_vars_subs(U_symbols):
    return {U(x,y,z):U(x) for U in U_symbols}
vars_subs = gen_vars_subs(E_symbols[0])
vars_subs = vars_subs | gen_vars_subs(H_symbols[0])
maxwell_diff_eqs = list_subs(maxwell_diff_eqs, vars_subs)

In [1067]:
# maxwell_diff_eqs[0]
sol = dsolve_system(list_subs(maxwell_diff_eqs, vars_subs), 
              funcs=[E_symbols[0][1](x),E_symbols[0][2](x),H_symbols[0][1](x),H_symbols[0][2](x)], t=x)[0]

In [1070]:
from itertools import chain, product

def gen_vars_subs_2(U_symbols):
    return {U(x, y, z): U(x) for U in list(chain.from_iterable(U_symbols))}
list_subs(maxwell_diff_eqs, gen_vars_subs_2([E_symbols[0],H_symbols[0]]))

[Eq(-Derivative(E^z_0(x), x) + I*mu*omega*H^y_0(x)/c + I*omega*H^y_0(x)*Derivative(phi(z), z)**2/(c*epsilon), 0),
 Eq(Derivative(E^y_0(x), x) + I*mu*omega*H^z_0(x)/c, 0),
 Eq(-Derivative(H^z_0(x), x) + I*epsilon*omega*E^y_0(x)/c + I*omega*E^y_0(x)*Derivative(phi(z), z)**2/(c*mu), 0),
 Eq(Derivative(H^y_0(x), x) + I*epsilon*omega*E^z_0(x)/c, 0)]

In [1080]:
gen_vars_subs_2([E_symbols[0], H_symbols[0], E_symbols[1], H_symbols[1]])

{E^x_0(x, y, z): E^x_0(x),
 E^y_0(x, y, z): E^y_0(x),
 E^z_0(x, y, z): E^z_0(x),
 H^x_0(x, y, z): H^x_0(x),
 H^y_0(x, y, z): H^y_0(x),
 H^z_0(x, y, z): H^z_0(x),
 E^x_1(x, y, z): E^x_1(x),
 E^y_1(x, y, z): E^y_1(x),
 E^z_1(x, y, z): E^z_1(x),
 H^x_1(x, y, z): H^x_1(x),
 H^y_1(x, y, z): H^y_1(x),
 H^z_1(x, y, z): H^z_1(x)}

In [882]:
[E_symbols[1](x),E_symbols[2](x),H_symbols[1](x),H_symbols[2](x)]

[E^y_0(x), E^z_0(x), H^y_0(x), H^z_0(x)]

In [894]:
sol[0].subs({epsilon*mu + Derivative(phi(z), z)**2:eta(z)})

Eq(E^y_0(x), I*C1*mu*exp(-omega*x*sqrt(eta(z))/c)/sqrt(eta(z)) - I*C2*mu*exp(omega*x*sqrt(eta(z))/c)/sqrt(eta(z)))

In [895]:
#gamma = Function('gamma')
sol = list_subs(sol,{epsilon*mu + Derivative(phi(z), z)**2:eta(z)})

In [807]:
funcs = [E_symbols[1](x),E_symbols[2](x),H_symbols[1](x),H_symbols[2](x)]
checkodesol(maxwell_diff_eqs,sol, func=funcs2)

In [896]:
list_subs(maxwell_diff_eqs, {eta(z):sqrt(epsilon*mu + beta**2)})

[Eq(-Derivative(E^z_0(x), x) + I*mu*omega*H^y_0(x)/c + I*omega*H^y_0(x)*Derivative(phi(z), z)**2/(c*epsilon), 0),
 Eq(Derivative(E^y_0(x), x) + I*mu*omega*H^z_0(x)/c, 0),
 Eq(-Derivative(H^z_0(x), x) + I*epsilon*omega*E^y_0(x)/c + I*omega*E^y_0(x)*Derivative(phi(z), z)**2/(c*mu), 0),
 Eq(Derivative(H^y_0(x), x) + I*epsilon*omega*E^z_0(x)/c, 0)]

In [897]:
maxwell_diff_eqs[0].subs({diff(phi(z),z)**2:beta})

Eq(I*beta*omega*H^y_0(x)/(c*epsilon) - Derivative(E^z_0(x), x) + I*mu*omega*H^y_0(x)/c, 0)

In [898]:
maxwell_diff_eqs[0]

Eq(-Derivative(E^z_0(x), x) + I*mu*omega*H^y_0(x)/c + I*omega*H^y_0(x)*Derivative(phi(z), z)**2/(c*epsilon), 0)

In [899]:
from sympy.utilities.iterables import iterable
from sympy.core.function import (Derivative, AppliedUndef, diff)

def my_checksysodesol(eqs, sols, func=None):
    r"""
    Substitutes corresponding ``sols`` for each functions into each ``eqs`` and
    checks that the result of substitutions for each equation is ``0``. The
    equations and solutions passed can be any iterable.

    This only works when each ``sols`` have one function only, like `x(t)` or `y(t)`.
    For each function, ``sols`` can have a single solution or a list of solutions.
    In most cases it will not be necessary to explicitly identify the function,
    but if the function cannot be inferred from the original equation it
    can be supplied through the ``func`` argument.

    When a sequence of equations is passed, the same sequence is used to return
    the result for each equation with each function substituted with corresponding
    solutions.

    It tries the following method to find zero equivalence for each equation:

    Substitute the solutions for functions, like `x(t)` and `y(t)` into the
    original equations containing those functions.
    This function returns a tuple.  The first item in the tuple is ``True`` if
    the substitution results for each equation is ``0``, and ``False`` otherwise.
    The second item in the tuple is what the substitution results in.  Each element
    of the ``list`` should always be ``0`` corresponding to each equation if the
    first item is ``True``. Note that sometimes this function may return ``False``,
    but with an expression that is identically equal to ``0``, instead of returning
    ``True``.  This is because :py:meth:`~sympy.simplify.simplify.simplify` cannot
    reduce the expression to ``0``.  If an expression returned by each function
    vanishes identically, then ``sols`` really is a solution to ``eqs``.

    If this function seems to hang, it is probably because of a difficult simplification.

    Examples
    ========

    >>> from sympy import Eq, diff, symbols, sin, cos, exp, sqrt, S, Function
    >>> from sympy.solvers.ode.subscheck import checksysodesol
    >>> C1, C2 = symbols('C1:3')
    >>> t = symbols('t')
    >>> x, y = symbols('x, y', cls=Function)
    >>> eq = (Eq(diff(x(t),t), x(t) + y(t) + 17), Eq(diff(y(t),t), -2*x(t) + y(t) + 12))
    >>> sol = [Eq(x(t), (C1*sin(sqrt(2)*t) + C2*cos(sqrt(2)*t))*exp(t) - S(5)/3),
    ... Eq(y(t), (sqrt(2)*C1*cos(sqrt(2)*t) - sqrt(2)*C2*sin(sqrt(2)*t))*exp(t) - S(46)/3)]
    >>> checksysodesol(eq, sol)
    (True, [0, 0])
    >>> eq = (Eq(diff(x(t),t),x(t)*y(t)**4), Eq(diff(y(t),t),y(t)**3))
    >>> sol = [Eq(x(t), C1*exp(-1/(4*(C2 + t)))), Eq(y(t), -sqrt(2)*sqrt(-1/(C2 + t))/2),
    ... Eq(x(t), C1*exp(-1/(4*(C2 + t)))), Eq(y(t), sqrt(2)*sqrt(-1/(C2 + t))/2)]
    >>> checksysodesol(eq, sol)
    (True, [0, 0])

    """
    def _sympify(eq):
        return list(map(sympify, eq if iterable(eq) else [eq]))
    eqs = _sympify(eqs)
    for i in range(len(eqs)):
        if isinstance(eqs[i], Equality):
            eqs[i] = eqs[i].lhs - eqs[i].rhs
    if func is not None:
        funcs = list(set(func))
    else:
        funcs = []
        for eq in eqs:
            derivs = eq.atoms(Derivative)
            func = set().union(*[d.atoms(AppliedUndef) for d in derivs])
            funcs.extend(func)
        funcs = list(set(funcs))

    if not all(isinstance(func, AppliedUndef) and len(func.args) == 1 for func in funcs)\
    and len({func.args for func in funcs})!=1:
        raise ValueError("func must be a function of one variable, not %s" % func)
    # commented out for working for multiple functions in sols
    # for sol in sols:
    #     if len(sol.atoms(AppliedUndef)) != 1:
    #         raise ValueError("solutions should have one function only")
    if len(funcs) != len({sol.lhs for sol in sols}):
        raise ValueError("number of solutions provided does not match the number of equations")
    dictsol = {}
    for sol in sols:
        func = list(sol.atoms(AppliedUndef))[0]
        if sol.rhs == func:
            sol = sol.reversed
        solved = sol.lhs == func and not sol.rhs.has(func)
        if not solved:
            rhs = solve(sol, func)
            if not rhs:
                raise NotImplementedError
        else:
            rhs = sol.rhs
        dictsol[func] = rhs
    checkeq = []
    for eq in eqs:
        for func in funcs:
            eq = sub_func_doit(eq, func, dictsol[func])
        ss = simplify(eq)
        if ss != 0:
            eq = ss.expand(force=True)
            if eq != 0:
                eq = sqrtdenest(eq).simplify()
        else:
            eq = 0
        checkeq.append(eq)
    if len(set(checkeq)) == 1 and list(set(checkeq))[0] == 0:
        return (True, checkeq)
    else:
        return (False, checkeq)    

In [838]:
my_checksysodesol(maxwell_diff_eqs, list_subs(sol,{eta(z):sqrt(epsilon*mu + diff(phi(z),z)**2)}), func=funcs)

NotImplementedError: 

In [900]:
maxwell_diff_eqs[0].subs(sqrt(epsilon*mu + diff(phi(z),z)**2), eta(z))

Eq(-Derivative(E^z_0(x), x) + I*mu*omega*H^y_0(x)/c + I*omega*H^y_0(x)*Derivative(phi(z), z)**2/(c*epsilon), 0)

In [901]:
list_subs(sol,{eta(z):sqrt(epsilon*mu + diff(phi(z),z)**2)})[0]

Eq(E^y_0(x), I*C1*mu*exp(-omega*x*(epsilon*mu + Derivative(phi(z), z)**2)**(1/4)/c)/(epsilon*mu + Derivative(phi(z), z)**2)**(1/4) - I*C2*mu*exp(omega*x*(epsilon*mu + Derivative(phi(z), z)**2)**(1/4)/c)/(epsilon*mu + Derivative(phi(z), z)**2)**(1/4))

In [872]:
maxwell_diff_eqs[2].lhs.collect(H_symbols[1](x)).simplify().subs({epsilon*mu + diff(phi(z),z)**2:eta(z)})

-Derivative(H^z_0(x, y, z), x) + I*epsilon*omega*E^y_0(x, y, z)/c + I*omega*E^y_0(x, y, z)*Derivative(phi(z), z)**2/(c*mu)

In [863]:
maxwell_diff_eqs = [eq.lhs.collect(H_symbols[1](x))
                    .simplify()
                    .subs({epsilon*mu + diff(phi(z),z)**2:eta(z)}) 
                    for eq in maxwell_diff_eqs]

In [871]:
maxwell_diff_eqs[2]

Eq(-Derivative(H^z_0(x, y, z), x) + I*epsilon*omega*E^y_0(x, y, z)/c + I*omega*E^y_0(x, y, z)*Derivative(phi(z), z)**2/(c*mu), 0)

(E^y_0(x),
 I*C1*mu*exp(-omega*x*sqrt(epsilon*mu + Derivative(phi(z), z)**2)/c)/sqrt(epsilon*mu + Derivative(phi(z), z)**2) - I*C2*mu*exp(omega*x*sqrt(epsilon*mu + Derivative(phi(z), z)**2)/c)/sqrt(epsilon*mu + Derivative(phi(z), z)**2))

In [980]:
expand(sol)

AttributeError: 'list' object has no attribute 'expand'

In [1081]:
sols[0]

NameError: name 'sols' is not defined